# AstroPredict — Notebook 01  
## Data Preparation Pipeline

This notebook prepares the raw SHARP solar magnetogram data for machine learning.
It loads multiple CSV files, cleans and standardizes them, creates temporal features,
generates flare prediction labels, and produces a final ML-ready dataset.

This notebook performs **data preparation only**.
No model training or prediction is done here.


## A0 — Environment Setup and Directory Configuration

### Objective
Set up Google Drive access and define file locations used throughout the notebook.

- Google Drive is mounted to enable persistent storage
- Required Python libraries are imported
- A directory is created to store the final processed dataset
- File paths for raw SHARP data and final output are defined




In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import os, shutil, subprocess

# Final clean output directory
BASE_DIR = Path("/content/drive/MyDrive/AstroPredict_Final")
BASE_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET = BASE_DIR / "ml_ready_dataset.csv"

# Runtime SHARP directory
SHARP_DIR = Path("/content/sharp_data/recovered_csvs")

print("SHARP runtime directory:", SHARP_DIR)
print("Final dataset will be saved to:", FINAL_DATASET)



SHARP runtime directory: /content/sharp_data/recovered_csvs
Final dataset will be saved to: /content/drive/MyDrive/AstroPredict_Final/ml_ready_dataset.csv


## A1 — Restore SHARP CSV Files

This cell restores SHARP CSV files into the runtime workspace if they are not already present.

- Defines paths for the compressed dataset, extraction directory, and recovered CSV folder
- Creates required directories if missing
- Copies the dataset archive from Google Drive
- Extracts the archive using 7-Zip
- Collects and consolidates all CSV files into a single directory
- Skips extraction if CSV files already exist


In [3]:
ZIP_PATH = Path("/content/sharp_data/zenodo_solar_flare_dataset.zip")
EXTRACT_BASE = Path("/content/sharp_data/extracted_7z")
EXTRACT_DIR = Path("/content/sharp_data/recovered_csvs")
DRIVE_ZIP = Path("/content/drive/MyDrive/AstroPredict_external/zenodo_solar_flare_dataset.zip")

EXTRACT_BASE.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if len(list(EXTRACT_DIR.glob("*.csv"))) == 0:
    print("⚠️ Runtime empty. Restoring SHARP data from Drive...")

    if not DRIVE_ZIP.exists():
        raise RuntimeError("Zenodo ZIP not found in Drive")

    shutil.copy(DRIVE_ZIP, ZIP_PATH)

    subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)
    subprocess.run(["7z", "x", str(ZIP_PATH), f"-o{EXTRACT_BASE}", "-y"], check=False)

    csv_files = list(EXTRACT_BASE.rglob("*.csv"))
    for f in csv_files:
        shutil.copy(f, EXTRACT_DIR / f.name)

    print(f"✅ Restore finished: {len(csv_files)} CSVs copied")
else:
    print("✅ SHARP CSVs already present")


⚠️ Runtime empty. Restoring SHARP data from Drive...
✅ Restore finished: 17 CSVs copied


## A1.1 — Verify Recovered CSV Files

Lists the recovered SHARP CSV files in the runtime directory along with their sizes.


In [4]:
!ls -lh /content/sharp_data/recovered_csvs


total 309M
-rw-r--r-- 1 root root 3.6M Jan 23 06:15 testing_data_C_24.csv
-rw-r--r-- 1 root root 4.5M Jan 23 06:15 testing_data_C_48.csv
-rw-r--r-- 1 root root 5.1M Jan 23 06:15 testing_data_C_72.csv
-rw-r--r-- 1 root root 3.6M Jan 23 06:15 testing_data_M_24.csv
-rw-r--r-- 1 root root 4.5M Jan 23 06:15 testing_data_M_48.csv
-rw-r--r-- 1 root root 3.6M Jan 23 06:15 testing_data_M5_24.csv
-rw-r--r-- 1 root root 4.5M Jan 23 06:15 testing_data_M5_48.csv
-rw-r--r-- 1 root root 5.1M Jan 23 06:15 testing_data_M5_72.csv
-rw-r--r-- 1 root root 5.1M Jan 23 06:15 testing_data_M_72.csv
-rw-r--r-- 1 root root  33M Jan 23 06:15 training_data_C_24.csv
-rw-r--r-- 1 root root  41M Jan 23 06:15 training_data_C_48.csv
-rw-r--r-- 1 root root  46M Jan 23 06:15 training_data_C_72.csv
-rw-r--r-- 1 root root  33M Jan 23 06:15 training_data_M_24.csv
-rw-r--r-- 1 root root  41M Jan 23 06:15 training_data_M_48.csv
-rw-r--r-- 1 root root  33M Jan 23 06:15 training_data_M5_24.csv
-rw-r--r-- 1 root root 150K Jan 23

## A1.2 — Load, Standardize, and Merge SHARP CSV Files

This cell loads all recovered SHARP CSV files and merges them into a single dataset.

- Skips one known corrupted file (`training_data_M5_48.csv`)
- Reads each valid CSV file individually
- Standardizes column names
- Unifies timestamp fields
- Ensures a valid active region identifier
- Selects required SHARP features and labels
- Renames columns to a consistent format
- Combines all cleaned files into one DataFrame


In [5]:
BAD_FILES = ["training_data_M5_48.csv"]
frames = []

for csv in SHARP_DIR.glob("*.csv"):
    if csv.name in BAD_FILES:
        continue

    df = pd.read_csv(csv)
    df.columns = [c.strip().lower() for c in df.columns]

    if "date__obs" in df.columns:
        df.rename(columns={"date__obs": "t_rec"}, inplace=True)
    if "date_obs" in df.columns:
        df.rename(columns={"date_obs": "t_rec"}, inplace=True)

    # IMPORTANT: never overwrite true HARPNUM
    if "harpnum" not in df.columns and "noaa_ar" in df.columns:
        df.rename(columns={"noaa_ar": "harpnum"}, inplace=True)

    base_cols = [
        "t_rec", "harpnum", "flare", "label",
        "usflux", "totusjh", "totusjz",
        "totpot", "r_value", "savncpp",
        "absnjzh", "meanalp"
    ]

    keep = [c for c in base_cols if c in df.columns]
    if len(keep) < 6:
        continue

    df = df[keep].copy()

    df = df.rename(columns={
        "t_rec": "timestamp",
        "harpnum": "harpnum",
        "flare": "flare_raw",
        "label": "flare_raw",
        "usflux": "USFLUX",
        "totusjh": "TOTUSJH",
        "totusjz": "TOTUSJZ",
        "totpot": "TOTPOT",
        "r_value": "R_VALUE",
        "savncpp": "SAVNCPP",
        "absnjzh": "ABSNJZH",
        "meanalp": "MEANALP"
    })

    frames.append(df)

sharp = pd.concat(frames, ignore_index=True)
print("Loaded SHARP rows:", len(sharp))
sharp.head()

Loaded SHARP rows: 1765269


,timestamp,harpnum,flare_raw,USFLUX,TOTUSJH,TOTUSJZ,TOTPOT,R_VALUE,SAVNCPP,ABSNJZH,MEANALP
0,2025-02-25 17:00:00,12768,N,-0.575266,-0.654323,-0.707283,-0.568348,0.613582,-0.916754,-0.978470,-0.237348
1,2025-02-25 17:12:00,12768,N,-0.574192,-0.654176,-0.708703,-0.569625,0.621855,-0.926384,-0.979552,-0.237288
2,2025-02-25 17:24:00,12768,N,-0.575631,-0.656392,-0.710504,-0.572246,0.623578,-0.945503,-0.969542,-0.237830
3,2025-02-25 17:36:00,12768,N,-0.577073,-0.657852,-0.712907,-0.573774,0.622889,-0.951383,-0.969496,-0.237841
4,2025-02-25 17:48:00,12768,N,-0.576406,-0.659400,-0.713725,-0.574640,0.628749,-0.942566,-0.979385,-0.237310


## A2 — Timestamp Cleaning and Active Region Filtering

This cell prepares the dataset for time-based processing.

- Converts the timestamp column to datetime format
- Removes rows with invalid timestamps
- Sorts observations by active region and time
- Removes active regions with fewer than 30 observations

After this step, only time-ordered active regions with sufficient data remain.


In [6]:
sharp["timestamp"] = pd.to_datetime(sharp["timestamp"], errors="coerce")
sharp = sharp.dropna(subset=["timestamp"])

sharp = sharp.sort_values(["harpnum", "timestamp"]).reset_index(drop=True)

counts = sharp["harpnum"].value_counts()
valid_ars = counts[counts >= 30].index
sharp = sharp[sharp["harpnum"].isin(valid_ars)].reset_index(drop=True)

print("After AR filtering:", sharp.shape)


After AR filtering: (1365478, 11)


## A3 — Flare Label Normalization

This cell converts the original flare labels into a binary format.

- Displays the original flare label distribution
- Maps flare labels to binary values:
  - `1` → flare occurred (`P`)
  - `0` → no flare (`N`)
- Prints the updated label distribution after conversion


In [7]:
print("Raw flare label distribution:")
print(sharp["flare_raw"].value_counts())

# Correct mapping: P = Positive flare, N = No flare
sharp["flare_label_24h"] = (
    sharp["flare_raw"]
    .astype(str)
    .str.upper()
    .isin(["P"])
).astype(int)

print("Corrected label distribution:")
print(sharp["flare_label_24h"].value_counts())


Raw flare label distribution:
flare_raw
P    785005
N    580473
Name: count, dtype: int64
Corrected label distribution:
flare_label_24h
1    785005
0    580473
Name: count, dtype: int64


## A4 — Temporal Feature Engineering

This cell creates time-based features for each solar active region.

- Computes 6-hour change (delta) for selected SHARP parameters
- Computes 6-hour rolling variability for selected SHARP parameters
- All calculations are performed separately for each active region

These features capture how magnetic parameters change over time.


In [8]:
df = sharp.copy()

# 6-hour deltas (12-min cadence → 30 steps)
for col in ["R_VALUE", "TOTUSJH", "TOTPOT"]:
    df[f"{col}_diff6h"] = df.groupby("harpnum")[col].diff(30)

# 6-hour rolling variability
for col in ["R_VALUE", "TOTUSJH"]:
    df[f"{col}_std6h"] = (
        df.groupby("harpnum")[col]
          .rolling(window=30, min_periods=1)
          .std()
          .reset_index(level=0, drop=True)
    )


## A5 — Multi-Horizon Flare Label Generation

This cell generates flare prediction labels for multiple future time windows.

- Sorts the data by active region and time
- Creates flare labels for:
  - 12-hour horizon
  - 6-hour horizon
- Labels indicate whether a flare occurs within the specified future window
- Ensures correct alignment of labels within each active region
- Verifies that no missing values are introduced


In [9]:
print("Generating labels (this takes ~1 minute)...")

# 1. Ensure correct sort order
df = df.sort_values(["harpnum", "timestamp"]).reset_index(drop=True)

def derive_horizon_series(base_series, hours):
    """
    base_series: flare_label_24h (pandas Series)
    returns: pandas Series of same length, explicitly aligned by index
    """
    # 12-min cadence → 5 steps per hour
    steps = int(hours * 5)
    values = base_series.values
    out = np.zeros(len(values), dtype=int)

    for i in range(len(values)):
        window_end = min(i + steps, len(values))
        if values[i:window_end].max() == 1:
            out[i] = 1

    # CRITICAL FIX: preserve original index
    return pd.Series(out, index=base_series.index)

# 2. Generate labels using transform (alignment-safe)
df["flare_label_12h"] = (
    df.groupby("harpnum")["flare_label_24h"]
      .transform(lambda s: derive_horizon_series(s, 12))
)

df["flare_label_6h"] = (
    df.groupby("harpnum")["flare_label_24h"]
      .transform(lambda s: derive_horizon_series(s, 6))
)

# 3. Sanity check
nan_counts = df[["flare_label_6h", "flare_label_12h"]].isna().sum()
print("NaNs after A5 (Must be 0):")
print(nan_counts)

if nan_counts.sum() > 0:
    raise RuntimeError("❌ CRITICAL ERROR: NaNs detected in labels! Alignment failed.")
else:
    print("✅ Labels generated successfully.")


Generating labels (this takes ~1 minute)...
NaNs after A5 (Must be 0):
flare_label_6h     0
flare_label_12h    0
dtype: int64
✅ Labels generated successfully.


## A6 — Data Quality Control and Cleanup

This cell applies final quality checks to the dataset.

- Checks for unexpected data loss before cleaning
- Removes rows with missing values
- Clips extreme values in SHARP features
- Reports the final dataset size after cleanup


In [10]:
# 1. SAFETY GUARD: row count before QC
initial_rows = len(df)
print(f"Rows before QC: {initial_rows}")

if initial_rows < 100_000:
    raise RuntimeError(
        f"❌ CATASTROPHIC DATA LOSS DETECTED. "
        f"Only {initial_rows} rows remain (Expected >1M). Check Cell A5."
    )

# 2. Drop NaNs (from diff / rolling warm-up)
df = df.dropna().reset_index(drop=True)

# 3. Clip outliers (3-sigma) — valid on normalized data
base_cols = [
    "USFLUX", "TOTUSJH", "TOTUSJZ", "TOTPOT",
    "R_VALUE", "SAVNCPP", "ABSNJZH", "MEANALP"
]

for col in base_cols:
    mu, sigma = df[col].mean(), df[col].std()
    df[col] = df[col].clip(mu - 3 * sigma, mu + 3 * sigma)

print(f"Final cleaned shape: {df.shape}")
print(f"Dropped {initial_rows - len(df)} rows during QC.")


Rows before QC: 1365478
Final cleaned shape: (1356208, 19)
Dropped 9270 rows during QC.


## A7 — Save Final Dataset

This cell saves the cleaned, ML-ready dataset to Google Drive and reports its size.

This dataset is used directly in:
- Model training
- Evaluation
- Real-time demonstration

In [11]:
df.to_csv(FINAL_DATASET, index=False)

print("✅ FINAL DATASET SAVED")
print(f"Path: {FINAL_DATASET}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")


✅ FINAL DATASET SAVED
Path: /content/drive/MyDrive/AstroPredict_Final/ml_ready_dataset.csv
Rows: 1356208
Columns: 19


## Notebook Summary

This notebook completed the full data preparation stage for the AstroPredict project.

In this notebook:
- Raw SHARP solar magnetogram CSV files were restored and verified
- Dataset inconsistencies across files were standardized
- Invalid and insufficient active-region data was removed
- Temporal features capturing 6-hour changes and variability were created
- Flare labels were converted into binary format
- Additional flare prediction labels for 6-hour and 12-hour horizons were generated
- Final quality checks were applied to remove invalid rows and extreme values
- A clean, ML-ready dataset was saved for downstream use

The output of this notebook is a single consolidated dataset containing:
- Standardized SHARP magnetic features
- Engineered temporal features
- Multi-horizon flare prediction labels

This dataset serves as the fixed input for all subsequent stages of the project, including:
- Model training
- Model evaluation
- Real-time simulation and API integration

Next step:
**Notebook-02 — Model Training**
